## This notebook postprocesses the results from previous notebooks

In [1]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd

In [2]:
# Define root directory and network directory path
# the study area should be in the network directory, and the network directory should be in the root directory

root_dir = Path(r"P:\11212759-001-brs\002-Kaartverhal_KEA\MS2\MS2proposal\data\paramaribo")
network_path = root_dir/ "static" / "network"
output_path = root_dir /"static" / "output_graph"

workfolder_test = root_dir / "outputs_ver04"/"outputs"

In [6]:
#Population demand for destinations
import pandas as pd
mode = "driving"

path = workfolder_test / "b_critical_locations.gpkg"
gdf = gpd.read_file(path)

gdf["category"].unique()
print(gdf.columns)

path2 = workfolder_test/mode/"cat_hexgrid_Critical_Infra_accessibility.gpkg"
gdf2 = gpd.read_file(path2)


path3 = workfolder_test/mode/"hazard_cat_hexgrid_Critical_Infra_accessibility.gpkg"
gdf3 = gpd.read_file(path3)

categories = ["supermarket", "school", "highway_bus_stop", "hospital"]

# Critical_Infra_id is a global unique facility id shared by all connected_{category} columns
population_demand = {}
for category in categories:
    col = f"connected_{category}"
    exploded = gdf2[[col, "population"]].dropna(subset=[col]).copy()
    exploded[col] = exploded[col].astype(str).str.split(";")
    exploded = exploded.explode(col)
    exploded[col] = exploded[col].astype(int)

    demand_by_id = exploded.groupby(col)["population"].sum()
    population_demand.update(demand_by_id.to_dict())

gdf["baseline_population_demand"] = gdf["Critical_Infra_id"].map(population_demand)


population_demand_hz = {}
for category in categories:
    col = f"connected_{category}"
    exploded1 = gdf3[[col, "population"]].dropna(subset=[col]).copy()
    exploded1[col] = exploded1[col].astype(str).str.split(";")
    exploded1 = exploded1.explode(col)
    exploded1[col] = exploded1[col].astype(int)

    demand_by_id = exploded1.groupby(col)["population"].sum()
    population_demand_hz.update(demand_by_id.to_dict())

gdf["hazard_population_demand"] = gdf["Critical_Infra_id"].map(population_demand_hz)

# Count people without access to each category in baseline and hazard
def people_without_access(df, category):
    col = f"connected_{category}"
    no_access = df[col].isna() | (df[col].astype(str).str.strip() == "")
    return df.loc[no_access, "population"].sum()

print("People without access per category (baseline vs hazard):")
for category in categories:
    baseline_no_access = people_without_access(gdf2, category)
    hazard_no_access = people_without_access(gdf3, category)
    change = hazard_no_access - baseline_no_access
    print(
        f"{category}: baseline={baseline_no_access:,.0f}, "
        f"hazard={hazard_no_access:,.0f}, change={change:+,.0f}"
    )

# Results accuracy check: for each category and scenario,
# total population should equal (population with access + population without access).
def population_access_accounting(df, category):
    col = f"connected_{category}"
    has_access = df[col].notna() & (df[col].astype(str).str.strip() != "")
    total_pop = df["population"].sum()
    access_pop = df.loc[has_access, "population"].sum()
    no_access_pop = df.loc[~has_access, "population"].sum()
    residual = total_pop - (access_pop + no_access_pop)
    return total_pop, access_pop, no_access_pop, residual

print("\nResults accuracy check (population accounting):")
check_rows = []
for category in categories:
    total_b, access_b, no_access_b, residual_b = population_access_accounting(gdf2, category)
    total_h, access_h, no_access_h, residual_h = population_access_accounting(gdf3, category)
    check_rows.append({
        "category": category,
        "baseline_total": total_b,
        "baseline_access": access_b,
        "baseline_no_access": no_access_b,
        "baseline_residual": residual_b,
        "baseline_status": "PASS" if abs(residual_b) < 1e-9 else "CHECK",
        "hazard_total": total_h,
        "hazard_access": access_h,
        "hazard_no_access": no_access_h,
        "hazard_residual": residual_h,
        "hazard_status": "PASS" if abs(residual_h) < 1e-9 else "CHECK"
    })

check_df = pd.DataFrame(check_rows)
display(check_df)

print("\nAre hazard columns identical?")
print("school == bus_stop:", gdf3["connected_school"].fillna("").equals(gdf3["connected_highway_bus_stop"].fillna("")))
print("school == hospital:", gdf3["connected_school"].fillna("").equals(gdf3["connected_hospital"].fillna("")))

display(gdf.head(5))


gdf.to_file(workfolder_test / "combined"/ f"{mode.capitalize()}_Critical_infrastructure_population_demand.gpkg", driver="GPKG")

Index(['population', 'type', 'category', 'Critical_Infra_id', 'geometry'], dtype='object')
People without access per category (baseline vs hazard):
supermarket: baseline=1,117, hazard=10,925, change=+9,808
school: baseline=1,117, hazard=10,925, change=+9,808
highway_bus_stop: baseline=1,117, hazard=10,925, change=+9,808
hospital: baseline=1,117, hazard=10,925, change=+9,808

Results accuracy check (population accounting):


,category,baseline_total,baseline_access,baseline_no_access,baseline_residual,baseline_status,hazard_total,hazard_access,hazard_no_access,hazard_residual,hazard_status
0,supermarket,271325.417464,270208.273802,1117.143663,0.0,PASS,271325.417464,260400.572462,10924.845003,0.0,PASS
1,school,271325.417464,270208.273802,1117.143663,0.0,PASS,271325.417464,260400.572462,10924.845003,0.0,PASS
2,highway_bus_stop,271325.417464,270208.273802,1117.143663,0.0,PASS,271325.417464,260400.572462,10924.845003,0.0,PASS
3,hospital,271325.417464,270208.273802,1117.143663,0.0,PASS,271325.417464,260400.572462,10924.845003,0.0,PASS



Are hazard columns identical?
school == bus_stop: False
school == hospital: False


,population,type,category,Critical_Infra_id,geometry,baseline_population_demand,hazard_population_demand
0,1,destination,supermarket,0,POINT (-55.21047 5.74797),NaN,NaN
1,1,destination,supermarket,1,POINT (-55.21096 5.74928),NaN,NaN
2,1,destination,supermarket,2,POINT (-55.24637 5.75093),NaN,NaN
3,1,destination,school,3,POINT (-55.21047 5.75224),NaN,NaN
4,1,destination,supermarket,4,POINT (-55.21096 5.75649),NaN,NaN


In [4]:
# ranking

mode = "driving"

path2 = workfolder_test/mode/"cat_hexgrid_Critical_Infra_accessibility.gpkg"
gdf2 = gpd.read_file(path2)

path3 = workfolder_test/mode/"hazard_cat_hexgrid_Critical_Infra_accessibility.gpkg"
gdf3 = gpd.read_file(path3)

path4 = workfolder_test/"flooded_hexgrid_100rp.gpkg"
gdf4 = gpd.read_file(path4)

# prefix non-key/geometry columns so joined columns stay traceable to their source gdf
gdf3_prefixed = gdf3.rename(columns={c: f"HZ_{c}" for c in gdf3.columns if c not in ("cell_id", "geometry")})
gdf4_prefixed = gdf4.rename(columns={c: f"FL_{c}" for c in gdf4.columns if c not in ("cell_id", "geometry")})

joined = gdf2.merge(gdf3_prefixed.drop(columns="geometry"), on="cell_id", how="left")
joined = joined.merge(gdf4_prefixed.drop(columns="geometry"), on="cell_id", how="left")

joined.head()

# Baseline_Score_15min: count of critical-service categories reachable within 15 minutes
time_cols = [c for c in joined.columns if c.startswith("time_nearest_") and c.endswith("_min")]
print("Category time columns used:", time_cols)

joined["Baseline_Score_10min"] = (joined[time_cols] < 10).sum(axis=1)

joined[["Baseline_Score_10min"] + time_cols].head()


time_cols_hz = [c for c in joined.columns if c.startswith("HZ_time_nearest_") and c.endswith("_min")]
print("Category time columns used:", time_cols_hz)

joined["HZ_Score_10min"] = (joined[time_cols_hz] < 10).sum(axis=1)

# Per-category Yes/No access: a row has access if connected_{category} / HZ_connected_{category} is not empty
connected_cols = [c for c in joined.columns if c.startswith("connected_")]

for col in connected_cols:
    category = col[len("connected_"):]

    has_access = joined[col].notna() & (joined[col].astype(str).str.strip() != "")
    joined[f"Access_{category}_baseline"] = np.where(has_access, "Yes", "No")

    hz_col = f"HZ_{col}"
    if hz_col in joined.columns:
        hz_has_access = joined[hz_col].notna() & (joined[hz_col].astype(str).str.strip() != "")
        joined[f"Access_{category}_hazard"] = np.where(hz_has_access, "Yes", "No")

access_cols = [c for c in joined.columns if c.startswith("Access_")]
joined[access_cols].head()


joined[["HZ_Score_10min"] + time_cols_hz].head()

joined.to_file(workfolder_test / "combined" / f"{mode.capitalize()}_combined_results.gpkg", driver="GPKG")

Category time columns used: ['time_nearest_supermarket_min', 'time_nearest_school_min', 'time_nearest_highway_bus_stop_min', 'time_nearest_hospital_min']
Category time columns used: ['HZ_time_nearest_supermarket_min', 'HZ_time_nearest_school_min', 'HZ_time_nearest_highway_bus_stop_min', 'HZ_time_nearest_hospital_min']
